1. Importing Necessary Libraries

In [1]:
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
print('pandas', pd.__version__)

pandas 2.2.3


2. Configuration 

In [ ]:
DATA_ROOT = '.'  # root folder where annotations and images are located
ANNOTATION_FILES = [
    '../../annotations/new_annotations.csv',
    '../../annotations/adv_annotations/Annotations_ANN_fgsm.csv',
    '../../annotations/adv_annotations/Annotations_CNN_fgsm.csv',
]
""" ANNOTATION_FILES = [
    '../../annotations/new_annotations.csv',
    '../../annotations/adv_annotations/Annotations_ANN_fgsm.csv',
    '../../annotations/adv_annotations/Annotations_ANN_pgd.csv',
    '../../annotations/adv_annotations/Annotations_CNN_fgsm.csv',
    '../../annotations/adv_annotations/Annotations_CNN_pgd.csv',
    '../../annotations/adv_annotations/Annotations_RNN_fgsm.csv',
    '../../annotations/adv_annotations/Annotations_RNN_pgd.csv',
] """
OUT_FILENAME = '../../annotations/train_detector_annotations.csv'  
#OUT_FILENAME = 'unkonwn_adversarial_detector_annotations.csv'  
SEED = 42

3. Robust CSV reader, auto-detect delimiter and parse annotation files

In [3]:
# Helpers to read annotation CSVs robustly and return a cleaned DataFrame.

POSSIBLE_DELIMS = [';', ',', '\t']

def detect_delim_firstline(path):
    """
    Detect the most likely delimiter used in a text CSV file by inspecting the first line.
    
    Args:
        path: Path to the CSV file.
    Returns:
        The detected delimiter as one of ';', ',' or '\\t'. Falls back to ','.
    """
    # Read just the first line to decide which delimiter appears
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        first = f.readline()
    for d in POSSIBLE_DELIMS:
        # For tab we check for the actual tab character
        if d == '\t':
            if '\t' in first:
                return '\t'
        else:
            if d in first:
                return d
    # Default fallback
    return ','

def read_annotation_csv(path):
    """
    Read an annotation CSV robustly: auto-detect delimiter, parse into a DataFrame,
    and handle cases where headers/rows are stored as a single semicolon-delimited cell.

    Behavior:
      - Detects delimiter from the first line (supports ';', ',' and '\\t').
      - Uses pandas.read_csv with the detected separator.
      - If pandas fails, falls back to reading lines and splitting by ';'.
      - If the resulting DataFrame has a single column which itself contains delimited columns
        (common when a CSV was written in a single cell), it attempts to split into separate columns.

    Args:
        path: Path to the annotation CSV file.
    Returns:
        pd.DataFrame with parsed columns (all values are strings).
    Raises:
        FileNotFoundError: If the file does not exist.
    """
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Annotation file not found: {path}')
    
    # Auto-detect delimiter and choose sep for pandas
    delim = detect_delim_firstline(path)
    sep = '\t' if delim == '\t' else delim
    try:
        # Use engine='python' for more flexible parsing of mixed delimiters
        df = pd.read_csv(path, sep=sep, engine='python', dtype=str)
    except Exception as e:
        # Fallback: read raw lines and split on semicolon — this is a best-effort recovery
        lines = open(path, 'r', encoding='utf-8', errors='ignore').read().splitlines()
        rows = [l.split(';') for l in lines if l.strip()]
        df = pd.DataFrame(rows)
    # Normalize column names (strip whitespace)
    df.columns = [c.strip() for c in df.columns]

    # If the file parsed into a single column, attempt to split values inside that column
    if len(df.columns) == 1:
        col0 = df.columns[0]
        # If the header itself contains semicolon-separated names, use them as columns
        if ';' in col0:
            new_cols = [c.strip() for c in col0.split(';')]
            split_rows = df[col0].str.split(';', expand=True)
            split_rows.columns = new_cols[:split_rows.shape[1]]
            df = split_rows
        else:
            # Otherwise split each row by semicolon and return a DataFrame with default integer column names
            split_rows = df[col0].str.split(';', expand=True)
            df = split_rows
    return df

4. Read & normalize all annotation files — merge into a single DataFrame

In [4]:
# normalize commonly-used column names, add a `source` column (file stem), and
# concatenate individual DataFrames into a single merged DataFrame.

all_dfs = []
for ann in ANNOTATION_FILES:
    # Prefer the annotation file relative to DATA_ROOT if it exists there,
    # otherwise use the given path (which may be absolute or relative to cwd).
    ann_path = os.path.join(DATA_ROOT, ann) if not os.path.isabs(ann) and os.path.exists(os.path.join(DATA_ROOT, ann)) else ann
    print('\nReading', ann_path)
    try:
        df = read_annotation_csv(ann_path)
    except Exception as e:
        # If a file fails to parse, print the error and continue with the next file
        print('ERROR reading', ann_path, e)
        continue

    # Keep the file stem as 'source' so we can trace back where rows came from
    source = Path(ann).stem
    df = df.copy()   # work on a copy to avoid side effects

    # Detect the image path/id column among common candidate names.
    # If none of the expected names appear, fall back to the first column.
    img_col = None
    for cand in ['image_id','image','filename','file','path','image_path','img_path','Filepath','fileName','ImageId']:
        if cand in df.columns:
            img_col = cand; break
    if img_col is None:
        img_col = df.columns[0]
        print(f'Using first column as image column for {ann}:', img_col)

    # Normalize the image column name to 'image_id' for consistency downstream
    df = df.rename(columns={img_col: 'image_id'})
    
    
    # Normalize other commonly used column names to canonical names:
    # - label  <- any of ['label', 'class', 'orig_class', 'y', 'target']
    # - isNight <- any of ['isNight', 'is_night', 'night']
    # - split <- any of ['split','set','subset']
    for cand in ['label','class','orig_class','y','target']:
        if cand in df.columns:
            df = df.rename(columns={cand:'label'}); break
    for cand in ['isNight','is_night','night']:
        if cand in df.columns:
            df = df.rename(columns={cand:'isNight'}); break
    for cand in ['split','set','subset']:
        if cand in df.columns:
            df = df.rename(columns={cand:'split'}); break
        
    # Ensure image_id is a stripped string and normalize path separators
    df['image_id'] = df['image_id'].astype(str).str.strip()
    df['image_id'] = df['image_id'].apply(lambda x: os.path.normpath(x) if isinstance(x,str) else x)

    # Add provenance column so we know the origin of each row
    df['source'] = source

    # Collect dataframe for later concatenation
    all_dfs.append(df)

# Concatenate all individual DataFrames into one merged DataFrame.
# If no valid DataFrames were loaded, create an empty DataFrame with no columns.
merged = pd.concat(all_dfs, ignore_index=True, sort=False)
print('\nMerged rows:', len(merged))
print('Columns after merge:', merged.columns.tolist())


Reading .\../../annotations/new_annotations.csv

Reading .\../../annotations/adv_annotations/Annotations_ANN_fgsm.csv

Reading .\../../annotations/adv_annotations/Annotations_CNN_fgsm.csv

Merged rows: 155478
Columns after merge: ['image_id', 'label', 'isNight', 'split', 'source']


5. Fix semicolon-embedded image_id rows & normalize image_path

In [5]:
# fills missing canonical columns (label, isNight, split) when possible, and creates a normalized image_path.
def fix_semicolon_rows(df):
    """
    Inspect the 'image_id' column for semicolon-delimited compound entries and split them into
    separate fields when detected. This handles cases like:
        "processed_images\\train\\img_0.jpg;1;0;train"
    which should map to image_id, label, isNight, split.

    Args:
        df: DataFrame that must contain an 'image_id' column (strings).
    Returns:
        A new DataFrame with corrected fields. Helper temporary columns are removed.
    """

    df = df.copy()
    if 'image_id' not in df.columns:
        # Nothing to do if image_id is missing
        return df
    
    # Find entries where image_id contains semicolons
    mask = df['image_id'].str.contains(';', na=False)
    if mask.any():
        print('Found rows where image_id contains semicolons - attempting to split into fields... Count:', mask.sum())
        parts = df.loc[mask, 'image_id'].str.split(';', expand=True)
        for i, col in enumerate(['image_id_part','label_part','isNight_part','split_part']):
            if i < parts.shape[1]:
                df.loc[mask, col] = parts[i].astype(str).str.strip()
        df.loc[mask & df['image_id_part'].notna(), 'image_id'] = df.loc[mask, 'image_id_part']

        # If canonical columns are missing or entirely NaN, fill them from extracted parts
        if 'label' not in df.columns or df['label'].isna().all():
            if 'label_part' in df.columns:
                df['label'] = df['label_part']
        if 'isNight' not in df.columns or df['isNight'].isna().all():
            if 'isNight_part' in df.columns:
                df['isNight'] = df['isNight_part']
        if 'split' not in df.columns or df['split'].isna().all():
            if 'split_part' in df.columns:
                df['split'] = df['split_part']

        # Drop temporary helper columns
        for c in ['image_id_part','label_part','isNight_part','split_part']:
            if c in df.columns:
                df = df.drop(columns=[c])
    else:
        print('No semicolon-embedded image_id rows found')
    return df

# Apply the fixer to the merged DataFrame
merged = fix_semicolon_rows(merged)

# Create a normalized image_path column (keeps platform-appropriate separators)
merged['image_path'] = merged['image_id'].apply(lambda x: os.path.normpath(x) if isinstance(x,str) else x)
merged[['image_id','image_path','source']].head(10)

No semicolon-embedded image_id rows found


,image_id,image_path,source
0,..\..\datasets\processed_images\train\img_0.jpg,..\..\datasets\processed_images\train\img_0.jpg,new_annotations
1,..\..\datasets\processed_images\train\img_1.jpg,..\..\datasets\processed_images\train\img_1.jpg,new_annotations
2,..\..\datasets\processed_images\train\img_2.jpg,..\..\datasets\processed_images\train\img_2.jpg,new_annotations
3,..\..\datasets\processed_images\train\img_3.jpg,..\..\datasets\processed_images\train\img_3.jpg,new_annotations
4,..\..\datasets\processed_images\train\img_4.jpg,..\..\datasets\processed_images\train\img_4.jpg,new_annotations
5,..\..\datasets\processed_images\train\img_5.jpg,..\..\datasets\processed_images\train\img_5.jpg,new_annotations
6,..\..\datasets\processed_images\train\img_6.jpg,..\..\datasets\processed_images\train\img_6.jpg,new_annotations
7,..\..\datasets\processed_images\train\img_7.jpg,..\..\datasets\processed_images\train\img_7.jpg,new_annotations
8,..\..\datasets\processed_images\train\img_8.jpg,..\..\datasets\processed_images\train\img_8.jpg,new_annotations
9,..\..\datasets\processed_images\train\img_9.jpg,..\..\datasets\processed_images\train\img_9.jpg,new_annotations


6. Infer adv_label from source & create stratified split

In [6]:
# Infer adv_label: source == 'new_annotations' -> clean(0), else adv(1)
# (This line creates a new column 'adv_label' where rows coming from
# 'new_annotations' are treated as clean (0), and all others as adversarial (1).)
merged['adv_label'] = merged['source'].apply(lambda s: 0 if str(s).strip().lower() == 'new_annotations' else 1)
print('adv_label distribution:')
print(merged['adv_label'].value_counts())

# Create stratified split if not present or if user chooses to re-split
# (If 'split' is missing or all NaN, we create train/val/test = 0.8/0.1/0.1 using stratification by adv_label.)
if 'split' not in merged.columns or merged['split'].isna().all():
    print('Creating stratified split...')
    train_frac = 0.8; val_frac=0.1; test_frac=0.1
    train, temp = train_test_split(merged, test_size=(1-train_frac), random_state=SEED, stratify=merged['adv_label'])
    val, test = train_test_split(temp, test_size=test_frac/(test_frac+val_frac), random_state=SEED, stratify=temp['adv_label'])
    train['split']='train'; val['split']='val'; test['split']='test'
    merged = pd.concat([train, val, test], ignore_index=True)
else:
    # If a valid 'split' column already exists, keep it as-is and do not modify splits.
    print('Using existing split')

print('\nCounts per split and adv_label:')
print(merged.groupby('split')['adv_label'].value_counts())

adv_label distribution:
adv_label
1    103652
0     51826
Name: count, dtype: int64
Using existing split

Counts per split and adv_label:
split  adv_label
test   1            28866
       0            14433
train  1            61414
       0            30707
val    1            13372
       0             6686
Name: count, dtype: int64


7. Resolve image paths relative to DATA_ROOT and validate existence

In [7]:
# Resolve image_path relative to DATA_ROOT when possible


def resolve_path(p):
    # This helper tries several heuristics to turn a stored path into a real filesystem path:
    # 1) If p is an absolute path and exists -> return it
    # 2) If p is relative and exists under DATA_ROOT -> return DATA_ROOT/p
    # 3) If basename exists under DATA_ROOT -> return DATA_ROOT/basename
    # 4) If basename exists under DATA_ROOT/processed_images -> return that
    # 5) If path contains an 'adv_outputs' prefix, strip it and try under DATA_ROOT
    # 6) Otherwise return the original p (fallback)
    if p is None: return p
    p = str(p)

    # 1) Absolute path that exists
    if os.path.isabs(p) and os.path.exists(p):
        return p
    
    # 2) Path relative to DATA_ROOT (preserve subfolders)
    cand = os.path.join(DATA_ROOT, p)
    if os.path.exists(cand):
        return cand
    
    # 3) Try placing the basename directly under DATA_ROOT
    cand2 = os.path.join(DATA_ROOT, Path(p).name)
    if os.path.exists(cand2):
        return cand2
    
    # 4) Try common processed_images folder under DATA_ROOT
    cand3 = os.path.join(DATA_ROOT, 'processed_images', Path(p).name)
    if os.path.exists(cand3):
        return cand3
    
    # 5) Remove adv_outputs prefix if present and try under DATA_ROOT
    cand4 = os.path.join(DATA_ROOT, p.replace('adv_outputs\\', '').replace('adv_outputs/', ''))
    if os.path.exists(cand4):
        return cand4

    # 6) Fallback: return the original value (may be relative to current cwd or invalid)
    return p

# Apply the resolver to the image_path column
merged['image_path'] = merged['image_path'].apply(resolve_path)

# Compute the fraction of image paths that actually exist on disk (0..1)
exists_frac = merged['image_path'].apply(lambda x: isinstance(x,str) and os.path.exists(x)).mean()
print(f'Fraction of image paths that exist on disk: {exists_frac:.3f}')

# If any paths are missing, display the first 10 examples to help debugging
missing = merged[~merged['image_path'].apply(lambda x: isinstance(x,str) and os.path.exists(x))]
if len(missing)>0:
    print('Sample missing (first 10):')
    display(missing[['image_id','image_path','source','adv_label']].head(10))

Fraction of image paths that exist on disk: 0.333
Sample missing (first 10):


,image_id,image_path,source,adv_label
51826,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51827,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51828,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51829,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51830,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51831,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51832,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51833,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51834,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1
51835,adv_outputs\ANN_fgsm\processed_images\train\im...,adv_outputs\ANN_fgsm\processed_images\train\im...,Annotations_ANN_fgsm,1


8. Drop helper columns and save merged annotations CSV

In [8]:
# Identify which helper columns we want to remove before saving
to_drop = [c for c in ['source','image_path'] if c in merged.columns]

# If any of the helper columns are present, drop them and inform the user
if to_drop:
    print('Dropping columns before save:', to_drop)
    merged = merged.drop(columns=to_drop)

# Ensure the output directory exists, then write the CSV without the index column
outpath = os.path.join(DATA_ROOT, OUT_FILENAME)

# Save merged annotations to CSV (UTF-8). Index is not saved to keep file clean.
merged.to_csv(outpath, index=False)
print('Saved merged annotations to', outpath)

# Display the first rows for a quick visual confirmation in the notebook
merged.head()

Dropping columns before save: ['source', 'image_path']
Saved merged annotations to .\../../annotations/train_detector_annotations.csv


,image_id,label,isNight,split,adv_label
0,..\..\datasets\processed_images\train\img_0.jpg,1,0,train,0
1,..\..\datasets\processed_images\train\img_1.jpg,1,0,train,0
2,..\..\datasets\processed_images\train\img_2.jpg,1,0,train,0
3,..\..\datasets\processed_images\train\img_3.jpg,1,0,train,0
4,..\..\datasets\processed_images\train\img_4.jpg,1,0,train,0
